# Ballenas jorobadas en el Pacífico colombiano: fase 0

¿Hay suficientes datos? Este cuaderno descarga los avistamientos públicos de *Megaptera novaeangliae* desde **GBIF** y **OBIS** para el Pacífico colombiano, los limpia y los muestra en un mapa y en gráficas por año, mes y fuente.

Criterio de decisión: con **300 registros útiles o más** (fecha y coordenadas, en el mar) se sigue con el modelo de hábitat. Con menos, se amplía al rango de la población G (Panamá, Costa Rica, Ecuador).

Ejecuta todo con *Entorno de ejecución > Ejecutar todas*. No necesita GPU.

In [ ]:
!pip -q install cartopy folium
import requests, time, pandas as pd, numpy as np, matplotlib.pyplot as plt
import cartopy.crs as ccrs, cartopy.feature as cfeature, folium
from folium.plugins import HeatMap

ESPECIE = "Megaptera novaeangliae"
# Zona de estudio: Pacífico colombiano, incluida la isla de Malpelo
OESTE, ESTE, SUR, NORTE = -82.5, -77.0, 1.4, 7.4
WKT = f"POLYGON(({OESTE} {SUR},{ESTE} {SUR},{ESTE} {NORTE},{OESTE} {NORTE},{OESTE} {SUR}))"
ACENTO, TINTA, GRIS = "#0F6E7A", "#1C211D", "#9AA19B"
plt.rcParams.update({"font.family": "DejaVu Sans", "axes.spines.top": False,
                     "axes.spines.right": False, "axes.edgecolor": GRIS,
                     "axes.labelcolor": TINTA, "xtick.color": TINTA, "ytick.color": TINTA})

## 1. GBIF

In [ ]:
clave = requests.get("https://api.gbif.org/v1/species/match", params={"name": ESPECIE}).json()["usageKey"]
filas, offset = [], 0
while True:
    r = requests.get("https://api.gbif.org/v1/occurrence/search", params={
        "taxonKey": clave, "geometry": WKT, "hasCoordinate": "true",
        "hasGeospatialIssue": "false", "limit": 300, "offset": offset}).json()
    filas += r["results"]
    offset += 300
    if r["endOfRecords"] or offset >= 100000: break
gbif = pd.DataFrame([{
    "fuente": "GBIF", "id": f"gbif_{x['key']}",
    "lat": x.get("decimalLatitude"), "lon": x.get("decimalLongitude"),
    "fecha": x.get("eventDate"), "individuos": x.get("individualCount"),
    "conjunto": x.get("datasetName") or x.get("datasetKey"),
    "base": x.get("basisOfRecord")} for x in filas])
print(f"GBIF: {r['count']} registros en la zona, {len(gbif)} descargados")

## 2. OBIS

In [ ]:
filas, despues = [], None
while True:
    p = {"scientificname": ESPECIE, "geometry": WKT, "size": 5000}
    if despues: p["after"] = despues
    r = requests.get("https://api.obis.org/v3/occurrence", params=p).json()
    lote = r.get("results", [])
    if not lote: break
    filas += lote
    despues = lote[-1]["id"]
    if len(lote) < 5000: break
obis = pd.DataFrame([{
    "fuente": "OBIS", "id": f"obis_{x['id']}",
    "lat": x.get("decimalLatitude"), "lon": x.get("decimalLongitude"),
    "fecha": x.get("eventDate"), "individuos": x.get("individualCount"),
    "conjunto": x.get("dataset_id"), "base": x.get("basisOfRecord")} for x in filas])
print(f"OBIS: {len(obis)} registros descargados")

## 3. Limpieza
Se unen las dos fuentes, se quitan registros sin fecha, los duplicados entre fuentes (mismo día y ~100 m) y los puntos que caen en tierra.

In [ ]:
df = pd.concat([gbif, obis], ignore_index=True)
df["fecha"] = pd.to_datetime(df["fecha"].astype(str).str[:10], errors="coerce")
total = len(df)
df = df.dropna(subset=["lat", "lon", "fecha"])
df["año"], df["mes"] = df.fecha.dt.year, df.fecha.dt.month
df["clave"] = df.lat.round(3).astype(str) + df.lon.round(3).astype(str) + df.fecha.dt.date.astype(str)
df = df.sort_values("fuente").drop_duplicates("clave")

# Quitar puntos en tierra con la línea de costa de Natural Earth (10 m)
import shapely, cartopy.io.shapereader as shpreader
from shapely.ops import unary_union
tierra = unary_union(list(shpreader.Reader(shpreader.natural_earth("10m", "physical", "land")).geometries()))
tierra = tierra.intersection(shapely.box(OESTE - 1, SUR - 1, ESTE + 1, NORTE + 1))
en_tierra = shapely.contains_xy(tierra, df.lon.values, df.lat.values)
df = df[~en_tierra].copy()

print(f"Descargados: {total}")
print(f"Útiles (fecha + coordenadas, en el mar, sin duplicados): {len(df)}")
print("Por fuente:", df.fuente.value_counts().to_dict())
print(f"Años: {df.año.min()} a {df.año.max()}")
print()
print("VEREDICTO:", "SEGUIR con el modelo de hábitat" if len(df) >= 300
      else "AMPLIAR la zona al rango de la población G")
df.drop(columns="clave").to_csv("avistamientos_pacifico_co.csv", index=False)

## 4. Mapa de avistamientos

In [ ]:
fig = plt.figure(figsize=(9, 10), dpi=150)
ax = plt.axes(projection=ccrs.PlateCarree())
ax.set_extent([OESTE, ESTE, SUR, NORTE])
ax.add_feature(cfeature.NaturalEarthFeature("physical", "ocean", "10m"), facecolor="#E8F1F2")
ax.add_feature(cfeature.NaturalEarthFeature("physical", "land", "10m"), facecolor="#F2F1EC", edgecolor=GRIS, linewidth=0.6)
ax.add_feature(cfeature.BORDERS.with_scale("10m"), edgecolor=GRIS, linewidth=0.5, linestyle=":")
for f, c in [("GBIF", ACENTO), ("OBIS", "#E07A2E")]:
    s = df[df.fuente == f]
    ax.scatter(s.lon, s.lat, s=10, c=c, alpha=0.55, linewidths=0, label=f"{f} ({len(s)})", transform=ccrs.PlateCarree())
sitios = {"Bahía Solano": (-77.40, 6.22), "Nuquí / Tribugá": (-77.35, 5.70), "Bahía Málaga": (-77.35, 4.05),
          "Gorgona": (-78.18, 2.97), "Malpelo": (-81.60, 4.00), "Buenaventura": (-77.03, 3.88)}
for n, (x, y) in sitios.items():
    ax.annotate(n, (x, y), xytext=(6, 0), textcoords="offset points", fontsize=9, color=TINTA,
                transform=ccrs.PlateCarree(), va="center")
gl = ax.gridlines(draw_labels=True, linewidth=0.3, color=GRIS); gl.top_labels = gl.right_labels = False
ax.legend(loc="lower left", frameon=False)
ax.set_title(f"Ballena jorobada en el Pacífico colombiano · {len(df)} registros útiles", loc="left", fontsize=13, color=TINTA)
plt.savefig("mapa_avistamientos.png", bbox_inches="tight"); plt.show()

## 5. ¿Cuándo? Año y mes

In [ ]:
fig, (a1, a2) = plt.subplots(1, 2, figsize=(14, 4.5), dpi=150, gridspec_kw={"width_ratios": [2, 1]})
porano = df.groupby("año").size()
a1.bar(porano.index, porano.values, color=ACENTO, width=0.8)
a1.set_title("Registros por año", loc="left", color=TINTA)
meses = ["E","F","M","A","M","J","J","A","S","O","N","D"]
pormes = df.groupby("mes").size().reindex(range(1, 13), fill_value=0)
a2.bar(range(1, 13), pormes.values, color=[ACENTO if 5 <= m <= 12 else GRIS for m in range(1, 13)])
a2.set_xticks(range(1, 13), meses)
a2.set_title("Registros por mes (temporada mayo-diciembre)", loc="left", color=TINTA)
plt.tight_layout(); plt.savefig("tiempo_avistamientos.png", bbox_inches="tight"); plt.show()

## 6. ¿Quién los tomó? Fuentes y sesgo turístico
Si pocos conjuntos concentran casi todo y caen en zonas de avistamiento, el sesgo es fuerte y hay que corregirlo en el modelo.

In [ ]:
top = df.conjunto.astype(str).value_counts().head(10)
print(top.to_string())
cerca = sum(((df.lon - x)**2 + (df.lat - y)**2) ** 0.5 < 0.25 for n, (x, y) in sitios.items() if n != "Malpelo")
print(f"\nRegistros a menos de ~25 km de un sitio turístico: {int((cerca > 0).sum())} de {len(df)} "
      f"({(cerca > 0).mean():.0%})")

## 7. Mapa interactivo
Acerca y aleja; el mapa de calor muestra dónde se concentran los registros.

In [ ]:
m = folium.Map(location=[4.4, -78.8], zoom_start=7, tiles="CartoDB positron")
HeatMap(df[["lat", "lon"]].values.tolist(), radius=12, blur=15).add_to(m)
for _, r in df.sample(min(len(df), 800), random_state=0).iterrows():
    folium.CircleMarker([r.lat, r.lon], radius=2, color=ACENTO, fill=True, weight=0,
                        fill_opacity=0.6, tooltip=f"{r.fuente} · {r.fecha.date()}").add_to(m)
m.save("mapa_interactivo.html"); m

## 8. Descargar resultados

In [ ]:
from google.colab import files
for f in ["avistamientos_pacifico_co.csv", "mapa_avistamientos.png", "tiempo_avistamientos.png", "mapa_interactivo.html"]:
    files.download(f)